In [1]:
'''
task: classify syllogism validity with NL notation
models: gemma-2-2b-it, llama-3.2-3b-instruct, phi-3.5-mini-instruct
dataset: pfolio
evaluation: zero-shot + sef category
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


In [3]:
import pandas as pd

pfolio_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_kr_gold_train_sef.csv")

In [4]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, sef, ref_relation=None, source_knowledge=None):
  # define sef categories
  sef_disjunctive = r"""
  A disjunctive syllogism contains "∨" or "⊕". Here is an example:
  <PREMISES>All kids are young.
All toddlers are kids.
If someone is young, then they are not elderly.
All pirates are seafarers.
If Nancy is not a pirate, then Nancy is young.
If Nancy is not a toddler, then Nancy is a seafarer.</PREMISES>
  <CONCLUSION>Nancy is either both a pirate and a toddler, or neither a pirate nor a toddler.</CONCLUSION>
  """

  sef_complex = r"""
  A complex syllogism has more than 2 premises. Here is an example:
  <PREMISES>Aberdeen won the cup in the 2013 final.
Rangers won the cup in the 2014 final.
Aberdeen and Rangers are different teams.
Different teams cannot win the cup in the same year's final.</PREMISES>
  <CONCLUSION>Aberdeen has once won a cup.</CONCLUSION>
  """

  sef_categorical = r"""
  A categorical syllogism contains any word from the list ["all", "any", "some", "no", "few", "most", "none", "several"]. Here is an example:
  <PREMISES>Ordinary is an unincorporated community.
Located within Elliot County, Ordinary is on Kentucky Route 32.
Ordinary is located northwest of Sandy Hook.</PREMISES>
  <CONCLUSION>There is an unincorporated community located in Elliot County.</CONCLUSION>
  """

  sef_hypothetical = r"""
  A hypothetical syllogism is not disjunctive, complex or categorical. Here is an example:
  <PREMISES>No homework is fun.
Some reading is homework.</PREMISES>
  <CONCLUSION>Some reading is fun.</CONCLUSION>
  """

  sef_categories = dict()
  sef_categories["disjunctive"] = sef_disjunctive
  sef_categories["complex"] = sef_complex
  sef_categories["categorical"] = sef_categorical
  sef_categories["hypothetical"] = sef_hypothetical


  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags.
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  You are also given the category of the syllogism to help you understand it: {sef_categories[sef]}.
  Classify the conclusion as "T" if true, "F" if false or "U" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

In [5]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      conclusion = row["Conclusions - " + notation]
      premises = row["Premises - " + notation]
      sef_category = row["sef"]
      label = row["Truth Values"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, sef_category, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

In [6]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

In [9]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [10]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [11]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default')

*** Premises: 
 There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey.
Tom is not an Eastern wild turkey.
Tom is not an Osceola wild turkey.
Tom is not a Gould's wild turkey.
Tom is neither a Merriam's wild turkey nor a Rio Grande wild turkey.
Tom is a wild turkey.
*** Conclusion: 
 Tom is an Ocellated wild turkey.
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey.
Tom is not an Eastern wild turkey.
Tom is not an Osceola wild turkey.
Tom is not a Gould's wild turkey.
Tom is neither a Merriam's wild turkey nor a Rio Grande wild turkey.
Tom is a wild turkey.
*** Conclusion: 
 Tom is an Eastern wild turkey.
*** True Label: 
 F
*** Predicted Label: 
 T
*** Premises: 
 There are six types of w

In [12]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.5548172757475083
***** PRECISION *****
0.40613961979600743
***** RECALL *****
0.5675140571790608
***** F1 *****
0.4648512900661898


,Accuracy,Precision,Recall,F1
0,0.554817,0.40614,0.567514,0.464851


In [14]:
# try rag search with llama
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [15]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey.
Tom is not an Eastern wild turkey.
Tom is not an Osceola wild turkey.
Tom is not a Gould's wild turkey.
Tom is neither a Merriam's wild turkey nor a Rio Grande wild turkey.
Tom is a wild turkey.
*** Conclusion: 
 Tom is an Ocellated wild turkey.
*** True Label: 
 T
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey.
Tom is not an Eastern wild turkey.
Tom is not an Osceola wild turkey.
Tom is not a Gould's wild turkey.
Tom is neither a Merriam's wild turkey nor a Rio Grande wild turkey.
Tom is a wild turkey.
*** Conclusion: 
 Tom is an Eastern wild turkey.
*** True Label: 
 F
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

*** Premises: 
 Maggie Friedman is an American screenwriter and producer.
Maggie Friedman was the showrunner and executive producer of the lifetime television series Witches of East End.
Witches of East End is a fantasy-drama series.
Maggie Friedman produced and developed Eastwick.
Eastwick is a series by ABC.
*** Conclusion: 
 Maggie Friedman developed Witches of East End.
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 Shafaq-Asiman is a large complex of offshore geological structures in the Caspian Sea.
Baku is northwest of Shafaq-Asiman.
If place A is northwest of place B, then place B is southeast of place A.
*** Conclusion: 
 Baku is southeast of Shafaq-Asiman.
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 Shafaq-Asiman is a large complex of offshore geological structures in the Caspian Sea.
Baku is northwest of Shafaq-Asiman.
If place A is northwest of place B, then place B is southeast of place A.
*** Conclusion: 
 A large complex is southeast of Baku.
*** True Label: 
 T
*** Predicted Label: 
 None
*** Premises: 
 Shafaq-Asiman is a large complex of offshore geological structures in the Caspian Sea.
Baku is northwest of Shafaq-Asiman.
If place A is northwest of place B, then place B is southeast of place A.
*** Conclusion: 
 Baku is not northwest of offshore geological structures.
*** True Label: 
 F
*** Predicted Label: 
 None
Classification Report:                  precision    recall  f1-score   support

                      0.00      0.00      0.00         0
    T </output>       0.00      0.00      0.00         0
</output> tags.       0.00      0.00      0.00         0
              F       0.36      0.13      0.19        69
     F</output>       0.00      0.00  

In [16]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.09302325581395349
***** PRECISION *****
0.16666666666666666
***** RECALL *****
0.02407125423875246
***** F1 *****
0.038522815138733575


,Accuracy,Precision,Recall,F1
0,0.093023,0.166667,0.024071,0.038523


In [18]:
# try rag search with phi
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", device_map="auto")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [19]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default')

*** Premises: 
 There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey.
Tom is not an Eastern wild turkey.
Tom is not an Osceola wild turkey.
Tom is not a Gould's wild turkey.
Tom is neither a Merriam's wild turkey nor a Rio Grande wild turkey.
Tom is a wild turkey.
*** Conclusion: 
 Tom is an Ocellated wild turkey.
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 There are six types of wild turkeys: Eastern wild turkey, Osceola wild turkey, Gould’s wild turkey, Merriam’s wild turkey, Rio Grande wild turkey, and Ocellated wild turkey.
Tom is not an Eastern wild turkey.
Tom is not an Osceola wild turkey.
Tom is not a Gould's wild turkey.
Tom is neither a Merriam's wild turkey nor a Rio Grande wild turkey.
Tom is a wild turkey.
*** Conclusion: 
 Tom is an Eastern wild turkey.
*** True Label: 
 F
*** Predicted Label: 
 F</output>
*** Premises: 
 There are six t

In [20]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.26578073089701
***** PRECISION *****
0.2631033182503771
***** RECALL *****
0.1011814510032614
***** F1 *****
0.14389661316572197


,Accuracy,Precision,Recall,F1
0,0.265781,0.263103,0.101181,0.143897


In [21]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)